In [7]:
import sys
sys.path.insert(0, "../../../")

from src.agents import AgentDriver, AgentDriverConfig
from src.agents.utils import AgentConnectorConfig

import json
import re
from tqdm import tqdm

In [8]:
FORMATTED_TRACES_PATH = "../../../data/experiences_extraction/formatted_traces/2wikimultihopqa_dev/without_queryprep/part1(casual_answer_generator).json" # TO CHANGE

In [9]:
with open(FORMATTED_TRACES_PATH, 'r', encoding='utf-8') as fd:
    traces = json.loads(fd.read())

In [10]:
agentd_config = AgentDriverConfig(
    agent_config=AgentConnectorConfig(
        credentials={ 'model': 'qwen2.5:7b','host': 'localhost','port': 11437 }
    )
)

In [ ]:
# PCA + HDBSCAN

In [11]:
agent = AgentDriver.connect(agentd_config)

In [ ]:
EXPERIENCE_EXTRACTION_SYSTEM_PROMPT = '''
You are an expert AI analyst reviewing successful step sequences from an AI agent execution.
Your task is to extract reusable, actionable step-level experiences that can guide future agent executions.
Focus on identifying specific patterns, techniques, and decision points that contributed to success.
ANALYSIS FRAMEWORK:
• STEP PATTERN ANALYSIS: Identify the specific sequence of actions that led to success
• DECISION POINTS: Highlight critical decisions made during these steps
• TECHNIQUE EFFECTIVENESS: Analyze why specific approaches worked well
• REUSABILITY: Extract patterns that can be applied to similar scenarios
EXTRACTION PRINCIPLES:
• Focus on TRANSFERABLE TECHNIQUES and decision frameworks
• Frame insights as actionable guidelines and best practices
# Original Query
{query}
# Step Sequence Analysis
{step_sequence}
# Outcome
This step sequence was part of a successful trajectory.
OUTPUT FORMAT:
Generate 1-3 step-level success insights as JSON objects:
```json
[
{{
“when_to_use” : “Specific conditions when this success insight should be applied”,
“task_query” : “Identify the specific task query from the original trajectory that this success
experience is most closely related to. Extract the exact query text.”,
“generalized_query” : “Abstract the specific task query to create a more generalized task representation.”,
“query_template“ : “Pattern of the specific task query to create a more generalized task representation.”,
“experience” : “Detailed description of the successful step pattern and why it works”,
“generalized_experience” : “Abstract the specific experience to create a more generalized and reusable experience representation.”,
“experience_template” : “Pattern of the specific experience to create a more generalized and reusable experience representation.”,
“tags” : [“relevant", “keywords", “from", “the", “task", “query"],
“confidence” : 0.8,
“tools_used” : [“list", “of", “tools"]
}}
]
```
'''

EXPERIENCE_VERIFICATION_SYSTEM_PROMPT = '''
You are an expert AI analyst tasked with validating the quality and usefulness of extracted step-level experiences.
Your task is to assess whether the extracted experience is actionable, accurate, and valuable for future agent executions.
VALIDATION CRITERIA:
* ACTIONABILITY: Is the experience specific enough to guide future actions?
* ACCURACY: Does the experience correctly reflect the patterns observed?
* RELEVANCE: Is the experience applicable to similar future scenarios?
* CLARITY: Is the experience clearly articulated and understandable?
* UNIQUENESS: Does the experience provide novel insights or common knowledge?
# Experience to Validate
Condition: {condition}
Experience Content: {experience_content}
OUTPUT FORMAT:
Provide validation assessment:
```json
{{
“is_valid” : true/false,
“score” : 0.8,
“feedback” : “Detailed explanation of validation decision”,
“recommendations” : “Suggestions for improvement if applicable”
}}
```
Score should be between 0.0 (poor quality) and 1.0 (excellent quality).
Mark as invalid if score is below 0.3 or if there are fundamental issues with the experience.
'''

In [ ]:
extracted_experiences = dict()

for query_id in list(traces.keys())[:25]:
    print("QUERY ID: ", query_id)
    cur_experiences = []
    # experiences extraction
    filled_prompt = EXPERIENCE_EXTRACTION_SYSTEM_PROMPT.format(
        query=traces[query_id]['question'], 
        step_sequence='\n'.join([f"{i+1}. {step_info['step_query']}" for i, step_info in enumerate(traces[query_id]['search_steps'])]),
        context='\n'.join([f"{i+1}. {step_info['retrieved_info']}" for i, step_info in enumerate(traces[query_id]['search_steps'])])
    )
    print("FILLED PROMPT:\n", filled_prompt)
    response, stat = agent.generate(
        system_prompt=filled_prompt
    )
    print("RAW RESPONSE:\n", response)
    
    #raw_experiences = re.search("```json\n(.*?)\n```", response).group()[8:-4]
    #json_experiences = json.loads(raw_experiences)
    #print("FORMATTED RESPONSE:\n", json_experiences)
        

    # # experiences validation
    # for cur_experience in json_experiences:
    #     filled_prompt = EXPERIENCE_VERIFICATION_SYSTEM_PROMPT.format(
    #         condition=cur_experience['when_to_use'],
    #         experience_content=cur_experience['experience']
    #     )
    #     print("FILLED PROMPT:\n", filled_prompt)
    #     response, stat = agent.generate(
    #         system_prompt=filled_prompt
    #     )
    #     print("RAW RESPONSE:\n", response)
    #     raw_validitycheck = re.search("```json\n(.*?)\n```", response).group()[8:-4]
    #     json_validitycheck = json.loads(raw_validitycheck)
    #     print("FORMATTED RESPONSE:\n", json_validitycheck)

    #     cur_experience.append({'experience': cur_experience, 'validity_check': json_validitycheck})

    extracted_experiences[query_id] = response

In [ ]:
for query_id, experiences in extracted_experiences.items():
    print(query_id)
    print(experiences)

In [ ]:
agent.close_connection()

In [ ]:
example = '''
 ```json
[
    {
        "when_to_use": "When comparing the ages or birth dates of individuals in a query.",
        "task_query": "Who is younger, Aneta Hladíková or René Poupardin?",
        "generalized_query": "Comparing the age or birth date of two or more individuals.",
        "query_template": "Who is [younger/older] between [Person A] and [Person B]?",
        "experience": "The successful step pattern involved directly obtaining the dates of birth for both individuals. This approach worked because it provided a clear, quantitative basis to compare their ages accurately.",
        "generalized_experience": "To successfully answer age or birth date comparison queries, obtain the exact dates of birth for all relevant individuals and use these dates to make an accurate comparison.",
        "experience_template": "For [age/birth date] comparison queries, retrieve the specific [date of birth] for each individual involved and compare them directly.",
        "tags": ["age_comparison", "birth_date", "individuals"],
        "confidence": 0.8,
        "tools_used": ["search for dates of birth"]
    }
]
```
'''

In [1]:
from pydantic import BaseModel, Field
from typing import List, Literal


class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Тональность отзыва: положительная, отрицательная или нейтральная"
    )
    confidence: float = Field(
        description="Уверенность в анализе от 0.0 до 1.0",
        ge=0.0,  # больше или равно 0
        le=1.0   # меньше или равно 1
    )
    key_topics: List[str] = Field(
        description="Ключевые темы, упомянутые в отзыве",
        max_items=5
    )
    summary: str = Field(
        description="Краткое резюме отзыва в одном предложении",
        max_length=200
    )


/tmp/ipykernel_8895/2207226553.py:14: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  key_topics: List[str] = Field(


In [2]:
from langchain_core.output_parsers import JsonOutputParser

# Создаем парсер на основе нашей модели
parser = JsonOutputParser(pydantic_object=SentimentAnalysis)

print("Что генерирует парсер:")
print(parser.get_format_instructions())

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Что генерирует парсер:
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Mar

In [3]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="""Проанализируй отзыв: {review}

{format_instructions}

ТОЛЬКО JSON!""",
    input_variables=["review"], # Что должен предоставить пользователь
    partial_variables={         # Что заполняется автоматически
        "format_instructions": parser.get_format_instructions()
    }
)

In [4]:
formatted_text = prompt_template.format(review="Отличный товар!")

In [6]:
print(formatted_text)

Проанализируй отзыв: Отличный товар!

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any b

In [12]:
response, stat = agent.generate(system_prompt=formatted_text)

model='qwen2.5:7b' created_at='2026-09-30T08:48:50.24386793Z' done=True done_reason='stop' total_duration=28740567995 load_duration=28027952461 prompt_eval_count=491 prompt_eval_duration=161084197 eval_count=40 eval_duration=359553422 message=Message(role='assistant', content='{"sentiment":"positive","confidence":1.0,"key_topics":["отличный товар"],"summary":"Отзыв оставляет положительное впечатление о товаре."}', thinking=None, images=None, tool_name=None, tool_calls=None)


In [14]:
print(response)

{"sentiment":"positive","confidence":1.0,"key_topics":["отличный товар"],"summary":"Отзыв оставляет положительное впечатление о товаре."}


In [13]:
parsed_result = parser.invoke(response)
print(f"Тип результата: {type(parsed_result)}")
print("Структурированные данные:")
for key, value in parsed_result.items():
    print(f"  {key}: {value}")

Тип результата: <class 'dict'>
Структурированные данные:
  sentiment: positive
  confidence: 1.0
  key_topics: ['отличный товар']
  summary: Отзыв оставляет положительное впечатление о товаре.
